# Directorio de Establecimientos de Reclusión del Orden Nacional

Fuente: [datos.gov.co — 24zf-4cfu](https://www.datos.gov.co/Justicia-y-Derecho/Directorio-de-Establecimientos-de-Reclusi-n-de-Ord/24zf-4cfu/about_data) (API Socrata SODA).

In [ ]:
import re
import requests
import pandas as pd

URL = "https://www.datos.gov.co/resource/24zf-4cfu.json"

In [ ]:
def traer_datos(url: str, page_size: int = 1000) -> pd.DataFrame:
    """Descarga todos los registros paginando con $limit/$offset."""
    registros, offset = [], 0
    while True:
        r = requests.get(url, params={"$limit": page_size, "$offset": offset, "$order": ":id"}, timeout=60)
        r.raise_for_status()
        lote = r.json()
        registros.extend(lote)
        if len(lote) < page_size:
            break
        offset += page_size
    return pd.DataFrame(registros)

df_raw = traer_datos(URL)
print(df_raw.shape)
df_raw.head()

## Limpieza
Nombres de columnas legibles y coordenadas convertidas de grados-minutos-segundos (texto con formatos variados) a decimales.

In [ ]:
def dms_a_decimal(valor):
    """Convierte textos como -04°12'35.748\" o 05° 53' 30,96'' a grados decimales."""
    if pd.isna(valor):
        return None
    s = str(valor).strip().replace(",", ".")
    negativo = s.startswith("-") or s.upper().endswith(("S", "W", "O"))
    nums = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums:
        return None
    grados = nums[0] + (nums[1] / 60 if len(nums) > 1 else 0) + (nums[2] / 3600 if len(nums) > 2 else 0)
    return -grados if negativo else grados

df = df_raw.rename(columns={
    "direcci_n": "direccion",
    "clasificaci_n_art_11_ley": "clasificacion_art_11_ley_65",
    "nombre_establecimiento_sisipec": "nombre_establecimiento",
})
df["latitud"] = df["latitud"].apply(dms_a_decimal).round(6)
df["longitud"] = df["longitud"].apply(dms_a_decimal).round(6)
for c in df.select_dtypes(exclude="number"):
    df[c] = df[c].str.strip()

df.info()
df.head()

In [ ]:
# Coordenadas fuera del rango aproximado de Colombia (revisar manualmente)
fuera = df[~df["latitud"].between(-5, 14) | ~df["longitud"].between(-82, -66)]
fuera[["nombre_establecimiento", "latitud", "longitud"]]

In [ ]:
df.to_csv("establecimientos_reclusion.csv", index=False, encoding="utf-8-sig")

## Otras cárceles y policía (OpenStreetMap)

La base del INPEC solo incluye establecimientos del orden nacional. Para cárceles municipales/departamentales y sitios de policía (estaciones, subestaciones, CAI) no hay una fuente oficial nacional abierta, así que se toman de OpenStreetMap vía la API Overpass:

- `amenity=prison` → cárceles (incluye también las del INPEC).
- `amenity=police` → estaciones, CAI, comandos, etc.

**Ojo:** OSM es colaborativo; la cobertura es buena en ciudades y desigual en zonas rurales, y muchos registros no tienen nombre.

### Enlaces

- OpenStreetMap: https://www.openstreetmap.org
- API Overpass usada en el notebook: https://overpass-api.de/api/interpreter (respaldo: https://overpass.kumi.systems/api/interpreter)
- Ver la consulta en un mapa: https://overpass-turbo.eu (pegar la consulta de `traer_osm` y pulsar **Run**)
- Definición de las etiquetas:
  - Cárceles: https://wiki.openstreetmap.org/wiki/Tag:amenity%3Dprison
  - Policía: https://wiki.openstreetmap.org/wiki/Tag:amenity%3Dpolice
- Ver un lugar puntual: `https://www.openstreetmap.org/<id_fuente>`, por ejemplo https://www.openstreetmap.org/way/123456
- Licencia (ODbL): https://www.openstreetmap.org/copyright. Al publicar estos datos hay que citar "© colaboradores de OpenStreetMap" y compartir la tabla derivada con la misma licencia.


In [ ]:
import numpy as np

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

def traer_osm(amenity: str) -> pd.DataFrame:
    """Descarga de OpenStreetMap todos los elementos amenity=<amenity> dentro de Colombia."""
    query = f"""
    [out:json][timeout:180];
    area["ISO3166-1"="CO"][admin_level=2]->.co;
    nwr["amenity"="{amenity}"](area.co);
    out center tags;
    """
    ultimo_error = None
    for url in OVERPASS_URLS:
        try:
            r = requests.post(url, data={"data": query}, headers={"User-Agent": "query-collection-notebook"}, timeout=240)
            r.raise_for_status()
            elementos = r.json()["elements"]
            break
        except Exception as e:  # probar el siguiente servidor
            ultimo_error = e
    else:
        raise RuntimeError(f"No se pudo consultar Overpass: {ultimo_error}")

    filas = []
    for e in elementos:
        tags = e.get("tags", {})
        centro = e.get("center", {})
        filas.append({
            "osm_id": f"{e['type']}/{e['id']}",
            "amenity": amenity,
            "nombre": tags.get("name"),
            "operador": tags.get("operator"),
            "ciudad": tags.get("addr:city"),
            "latitud": e.get("lat", centro.get("lat")),
            "longitud": e.get("lon", centro.get("lon")),
        })
    return pd.DataFrame(filas)

osm_carceles = traer_osm("prison")
osm_policia = traer_osm("police")
print("Cárceles OSM:", len(osm_carceles), "| Policía OSM:", len(osm_policia))
osm_carceles.head()

### Separar cárceles del INPEC de las demás

Una cárcel de OSM se considera del INPEC si está a menos de `RADIO_INPEC_KM` de un establecimiento de la base oficial; las demás quedan como "otra cárcel" (municipales, departamentales, de menores, etc.).

In [ ]:
def distancia_km(lat1, lon1, lat2, lon2):
    """Haversine vectorizado: matriz de distancias (len(lat1) x len(lat2)) en km."""
    lat1, lon1, lat2, lon2 = map(np.radians, (np.asarray(lat1)[:, None], np.asarray(lon1)[:, None],
                                              np.asarray(lat2)[None, :], np.asarray(lon2)[None, :]))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

RADIO_INPEC_KM = 1.0

d = distancia_km(osm_carceles["latitud"], osm_carceles["longitud"], df["latitud"], df["longitud"])
osm_carceles["dist_inpec_km"] = d.min(axis=1).round(3)
osm_carceles["inpec_mas_cercano"] = df["nombre_establecimiento"].to_numpy()[d.argmin(axis=1)]
osm_carceles["es_inpec"] = osm_carceles["dist_inpec_km"] <= RADIO_INPEC_KM

print(osm_carceles["es_inpec"].value_counts())
otras_carceles = osm_carceles[~osm_carceles["es_inpec"]]
otras_carceles[["nombre", "operador", "ciudad", "latitud", "longitud", "dist_inpec_km"]].head(20)

### Tabla unificada

Una fila por lugar con `tipo` ∈ {`INPEC`, `Otra cárcel`, `Policía`}, lista para cruzar por distancia con otras bases (p. ej. transacciones).

In [ ]:
lugares = pd.concat([
    pd.DataFrame({
        "tipo": "INPEC",
        "fuente": "datos.gov.co 24zf-4cfu",
        "id_fuente": df["cod_establecimiento"],
        "nombre": df["nombre_establecimiento"],
        "departamento": df["departamento"],
        "ciudad": df["ciudad"],
        "latitud": df["latitud"],
        "longitud": df["longitud"],
    }),
    pd.DataFrame({
        "tipo": "Otra cárcel",
        "fuente": "OpenStreetMap",
        "id_fuente": otras_carceles["osm_id"],
        "nombre": otras_carceles["nombre"],
        "departamento": None,
        "ciudad": otras_carceles["ciudad"],
        "latitud": otras_carceles["latitud"],
        "longitud": otras_carceles["longitud"],
    }),
    pd.DataFrame({
        "tipo": "Policía",
        "fuente": "OpenStreetMap",
        "id_fuente": osm_policia["osm_id"],
        "nombre": osm_policia["nombre"],
        "departamento": None,
        "ciudad": osm_policia["ciudad"],
        "latitud": osm_policia["latitud"],
        "longitud": osm_policia["longitud"],
    }),
], ignore_index=True)

print(lugares["tipo"].value_counts())
lugares.to_csv("lugares_reclusion_policia.csv", index=False, encoding="utf-8-sig")
lugares.sample(10, random_state=1)

### Base solo de cárceles (INPEC + otras cárceles)

Sin sitios de policía. Es la base recomendada para cruzar con transacciones: el INPEC como fuente principal y las otras cárceles de OpenStreetMap marcadas aparte (revisarlas manualmente antes de usarlas).

In [ ]:
carceles = lugares[lugares["tipo"].isin(["INPEC", "Otra cárcel"])].reset_index(drop=True)

print(carceles["tipo"].value_counts())
carceles.to_csv("carceles_inpec_otras.csv", index=False, encoding="utf-8-sig")
carceles.head()

## Distancia de transacciones a cárceles

`distancia_a_carceles()` recibe cualquier tabla de transacciones con columnas de latitud y longitud en grados decimales y agrega, para cada transacción:

- `dist_inpec_km` y `inpec_cercana`: distancia a la cárcel del INPEC más cercana y su nombre.
- `dist_otra_carcel_km` y `otra_carcel_cercana`: lo mismo para las otras cárceles.
- `cerca_inpec` y `cerca_otra_carcel`: `True` si la distancia es menor o igual a `radio_km`.

Calcula por lotes para no llenar la memoria con bases de millones de filas. Las transacciones sin coordenadas quedan con distancia vacía y marca `False`.

In [ ]:
def distancia_a_carceles(trx: pd.DataFrame, carceles: pd.DataFrame,
                         col_lat: str = "latitud", col_lon: str = "longitud",
                         radio_km: float = 1.0, lote: int = 200_000) -> pd.DataFrame:
    """Agrega a cada transacción la distancia (km) a la cárcel más cercana de cada tipo."""
    out = trx.copy()
    grupos = {"inpec": carceles[carceles["tipo"] == "INPEC"],
              "otra_carcel": carceles[carceles["tipo"] == "Otra cárcel"]}
    lat = out[col_lat].to_numpy(dtype=float)
    lon = out[col_lon].to_numpy(dtype=float)
    validas = ~(np.isnan(lat) | np.isnan(lon))

    for sufijo, c in grupos.items():
        dist = np.full(len(out), np.nan)
        cercana = np.full(len(out), None, dtype=object)
        nombres = c["nombre"].fillna(c["id_fuente"]).to_numpy()
        idx = np.flatnonzero(validas)
        for ini in range(0, len(idx), lote):
            sel = idx[ini:ini + lote]
            d = distancia_km(lat[sel], lon[sel], c["latitud"], c["longitud"])
            j = d.argmin(axis=1)
            dist[sel] = d[np.arange(len(sel)), j]
            cercana[sel] = nombres[j]
        nombre_col = "inpec_cercana" if sufijo == "inpec" else "otra_carcel_cercana"
        out[f"dist_{sufijo}_km"] = dist.round(3)
        out[nombre_col] = cercana
        out[f"cerca_{sufijo}"] = dist <= radio_km  # NaN <= radio da False
    return out

### Con la base real de transacciones (.xlsx)

Celda comentada para cuando exista la tabla verdadera. Para usarla:

1. Cambiar `RUTA_TRX`, `HOJA` y los nombres de columna (`COL_ID`, `COL_MONTO`, `COL_LAT`, `COL_LON`) por los de la base.
2. Quitar los `#` (en VS Code: seleccionar la celda y `Ctrl+/`).
3. No correr la celda del ejemplo sintético; las tablas de resumen de abajo funcionan igual porque aquí se renombran las columnas a `id_transaccion` y `monto`.

**No subir a GitHub** la base real ni el notebook con resultados de datos reales (el repo es público).

In [ ]:
# RUTA_TRX = "transacciones.xlsx"   # ruta del archivo Excel
# HOJA = 0                           # nombre o número de la hoja
# COL_ID = "ID_TRANSACCION"          # columna identificadora de la transacción
# COL_MONTO = "VALOR"                # columna del monto
# COL_LAT = "LATITUD"                # columna de latitud (grados decimales)
# COL_LON = "LONGITUD"               # columna de longitud (grados decimales)
#
# # Requiere openpyxl (viene con Anaconda; si no: pip install openpyxl)
# trx = pd.read_excel(RUTA_TRX, sheet_name=HOJA)
# trx = trx.rename(columns={COL_ID: "id_transaccion", COL_MONTO: "monto",
#                           COL_LAT: "latitud", COL_LON: "longitud"})
# print(trx.shape)
#
# # Coordenadas a número (acepta coma decimal) y fuera de Colombia -> vacías
# for c in ["latitud", "longitud"]:
#     trx[c] = pd.to_numeric(trx[c].astype(str).str.strip().str.replace(",", "."), errors="coerce")
# malas = ~trx["latitud"].between(-5, 14) | ~trx["longitud"].between(-82, -66)
# print("Transacciones sin coordenadas válidas:", malas.sum())
# trx.loc[malas, ["latitud", "longitud"]] = np.nan
#
# # Si las coordenadas vienen en grados-minutos-segundos (texto), usar en su lugar:
# # trx["latitud"] = trx["latitud"].apply(dms_a_decimal)
# # trx["longitud"] = trx["longitud"].apply(dms_a_decimal)
#
# # Cuántas transacciones quedan dentro de cada radio, para elegir radio_km
# trx_dist = distancia_a_carceles(trx, carceles, radio_km=1.0)
# for r in [0.5, 1, 2, 5]:
#     print(f"{r} km -> INPEC: {(trx_dist['dist_inpec_km'] <= r).mean():.2%} | "
#           f"otras cárceles: {(trx_dist['dist_otra_carcel_km'] <= r).mean():.2%}")
#
# trx_dist.to_excel("transacciones_distancia_carceles.xlsx", index=False)
# trx_dist.sort_values("dist_inpec_km").head(10)

### Ejemplo con transacciones inventadas

**Estas transacciones son sintéticas**, solo para mostrar el resultado. Con datos reales, reemplazar `trx_ejemplo` por la base de transacciones e indicar los nombres de sus columnas de coordenadas en `col_lat` y `col_lon`.

In [ ]:
rng = np.random.default_rng(42)
n = 1_000

# 900 transacciones al azar en el Valle de Aburrá y 100 alrededor de la cárcel de Bello (CPMS BELLO)
bello = df.loc[df["nombre_establecimiento"].str.contains("CPMS BELLO", na=False), ["latitud", "longitud"]].iloc[0]
trx_ejemplo = pd.DataFrame({
    "id_transaccion": range(1, n + 1),
    "monto": rng.integers(10_000, 2_000_000, n),
    "latitud": np.r_[rng.uniform(6.10, 6.40, 900), bello["latitud"] + rng.normal(0, 0.01, 100)],
    "longitud": np.r_[rng.uniform(-75.70, -75.45, 900), bello["longitud"] + rng.normal(0, 0.01, 100)],
})

trx_dist = distancia_a_carceles(trx_ejemplo, carceles, radio_km=1.0)
print(trx_dist[["cerca_inpec", "cerca_otra_carcel"]].sum())
trx_dist.sort_values("dist_inpec_km").head(10)

In [ ]:
# Resumen por cárcel del INPEC: cuántas transacciones caen dentro del radio y cuánto suman
(trx_dist[trx_dist["cerca_inpec"]]
 .groupby("inpec_cercana")
 .agg(transacciones=("id_transaccion", "size"), monto_total=("monto", "sum"),
      dist_promedio_km=("dist_inpec_km", "mean"))
 .sort_values("transacciones", ascending=False))

In [ ]:
# Resumen por otra cárcel (OpenStreetMap): cuántas transacciones caen dentro del radio y cuánto suman
(trx_dist[trx_dist["cerca_otra_carcel"]]
 .groupby("otra_carcel_cercana")
 .agg(transacciones=("id_transaccion", "size"), monto_total=("monto", "sum"),
      dist_promedio_km=("dist_otra_carcel_km", "mean"))
 .sort_values("transacciones", ascending=False))